In [ ]:
import glob
from pathlib import Path

dataset_location = '../../data/RepLiQA/db_files/*_train.json'
dataset_files = glob.glob(dataset_location)

dataset_files, len(dataset_files)

from datasets import load_dataset

data = []
dataset_names = []

for dataset_file in dataset_files:
    dataset = load_dataset("json", data_files = dataset_file,  split='train')
    data.append(dataset)
    dataset_names.append(dataset_file.split("/")[-1].replace('_train.json',''))


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings


model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
    )

In [ ]:
from tqdm import tqdm

sample_size = 300

all_embeddings = []
all_tokens = []

for train_data_set in tqdm(data):
    dataset_embedding = []
    strings = []
    for dp in train_data_set.select(range(sample_size)):
        # for dx in dp:
        for key in [1,2]:
            strings.append(dp['messages'][key]['content'])
    dataset_embedding = embedding_model.embed_documents(strings)
    all_embeddings.append(dataset_embedding)

In [ ]:
import numpy as np
import torch
from MulticoreTSNE import MulticoreTSNE as TSNE
import pandas as pd

embeds = []

for emb in all_embeddings:
    embeds.append(np.array(emb))
all_embeds = np.concatenate(embeds)

len(all_embeds)

In [ ]:
from MulticoreTSNE import MulticoreTSNE as TSNE
import pandas as pd

tsne = TSNE(n_components=2,  verbose=True, n_jobs=80)

reduced_embeddings_tsne = tsne.fit_transform(all_embeds)


In [ ]:
labels = []
for datset_name in dataset_names:
    labels  = labels + [datset_name] * sample_size * 2


In [ ]:
import plotly.express as px

df = pd.DataFrame(reduced_embeddings_tsne, columns = ['x', 'y'])

df['labels'] = labels

fig = px.scatter(df, x='x', y='y', color = 'labels', opacity=0.4, title = 'Reduction with tSNE')
fig.show()

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(reduced_embeddings_tsne, columns = ['x', 'y'])
df['labels'] = labels
sample_labels = df['labels'].unique()
cmap = plt.colormaps.get_cmap('hsv').resampled(len(sample_labels))

fig, ax = plt.subplots(layout='tight')

for idx_sample_value, sample_value in enumerate(sample_labels):
    # Plot each sample data
    sub_data = df.loc[df['labels'] == sample_value]
    # Color tuple needs to be a list so that it applies only one color
    ax.scatter(x=sub_data['x'], y=sub_data['y'], s=50, c=[cmap(idx_sample_value)],
               alpha=0.5, marker = 'o', edgecolor='black', linewidth=0.5,
               label=f"{sample_value}")


ax.legend(sample_labels, ncol = 4, shadow = True, bbox_to_anchor=(1, -0.06))
fig.set_size_inches(12, 6)
plt.show()

fig.savefig("../figs/repliQA_embed.pdf",bbox_inches = 'tight',
    pad_inches = 0)